In [3]:
import pandas as pd

import Bio
from Bio import Align
from Bio import SeqIO
from Bio.Align.substitution_matrices import Array

import math
import numpy as np

In [2]:
aligner = Align.PairwiseAligner()

# ---------- Configure settings for local alignment ----------
aligner.mode = "local" # local alignment (Smith-Waterman)

# prevent aligner from adding gaps during the alignment -> the 9-mer ori query should be aligned exactly!
aligner.open_gap_score = -math.inf 
aligner.extend_gap_score = -math.inf


# average 'ori' sequence -> forms the query sequence for local alignent
# Deduced from weighted sequence logos in Fig 5. -> results of HUH-seq assay developed by Tompkins et al.
# see https://www.bioinformatics.org/sms/iupac.html for IUPAC codes for bases
ori_query = "HMDWNYTAC" 

In [4]:
bases = "ATGC"
alphabet = "ACGTRYSWKMHDN" #include "" or "" (edge case where download ref genome has ambiguous base code)


#---------- Custom Substitution Matrix ----------

# # Row (Target Base) x Column (Query Base)

default_scores = np.full((13, 13), -np.inf)
sub_matrix = Array(alphabet=alphabet, dims=2, data=default_scores)


# A(+1), C(+2): Final 2 bases sequences in 9-mer ori MUST be exactly "AC" 

sub_matrix["A", "A"] = 50.0 # must be a large value (avoid edge case where terminating sequence gives optimal alignment -> "AC" never matched so -inf penalty not applied)
sub_matrix["C", "C"] = 50.0

#

# T(-1):
sub_matrix.update({('A', 'T'): 0.9, ('C', 'T'): 1.0, ('G', 'T'): 0.8, ('T', 'T'): 6.8})

# Y(-2):
sub_matrix.update({('A', 'Y'): 1.4, ('C', 'Y'): 2.8, ('G', 'Y'): 0.8, ('T', 'Y'): 4.1})

# N(-3):
sub_matrix.update({('A', 'N'): 3.6, ('C', 'N'): 1.8, ('G', 'N'): 1.9, ('T', 'N'): 2.3})

# W(-4):
sub_matrix.update({('A', 'W'): 1.4, ('C', 'W'): 0.9, ('G', 'W'): 0.4, ('T', 'W'): 6.6})

# D(-5):
sub_matrix.update({('A', 'D'): 6.6, ('C', 'D'): 0.8, ('G', 'D'): 0.7, ('T', 'D'): 1.3})

# M(-6):
sub_matrix.update({('A', 'M'): 6.9, ('C', 'M'): 0.9, ('G', 'M'): 0.3, ('T', 'M'): 1.3})

# H(-7):
sub_matrix.update({('A', 'H'): 2.0, ('C', 'H'): 2.1, ('G', 'H'): 0.6, ('T', 'H'): 4.7})


aligner.substitution_matrix = sub_matrix

In [24]:
MASK_CHAR = "N"

def _local_alignment(virus_genome_str):
    # Returns: best alignments, best score, next-best alignments, next-best score
    optimal_score = aligner.score(virus_genome_str, ori_query)
    alignments = aligner.align(virus_genome_str, ori_query)

    # Mask regions covered by the optimal alignment(s); same length, so indices stay valid
    masked = list(virus_genome_str)
    for alignment in alignments:
        start = alignment.coordinates[0][0]
        end = alignment.coordinates[0][-1]
        masked[start:end] = [MASK_CHAR] * (end - start)
    masked_str = "".join(masked)

    next_score = aligner.score(masked_str, ori_query)
    next_alignments = aligner.align(masked_str, ori_query)

    return alignments, optimal_score, next_alignments, next_score


def _extract_candidates(genome_str, alignments):
    candidates = []
    for alignment in alignments:
        start = alignment.coordinates[0][0]
        end = alignment.coordinates[0][-1]
        # slice the ORIGINAL genome, not the masked one
        candidates.append(genome_str[start:end])
    return candidates


def obtain_ori_candidates(virus_genome_str):
    '''

    Once local alignment has been performed, converts results into list of ori candidates.

    Args:
        alignments: Pairwise alignment object
            -> result of local alignment
        virus_genome_str: str
            -> virus genome as a string (not fasta!)

        Returns:
        Ori candidates: [str]
    
    '''

    alignments, optimal_score, next_alignments, next_score = _local_alignment(virus_genome_str)

    ori_candidates = _extract_candidates(virus_genome_str, alignments)
    next_ori_candidates = _extract_candidates(virus_genome_str, next_alignments)

    return ori_candidates, optimal_score, next_ori_candidates, next_score

In [ ]:
genome = """
TAATATTACCGCCCATCTTTAGCCGAGGGACCATGCCCAACCGGTCCCGAGGCGCCCGGACCCCCCAAACACTAGGGGTT
GCACCCCGCAATATGATTTTCGAATCTGACTGCCATCTTCCTTTGTTTTGTCTTTATAAGACTAGATCGCACTAGTACGC
TATGGCGGAGTACCCTCAGTCTGCTTTCGTTTTATCTGGTGCGATTCCACATCAAAGCAAGGACGAAGGCTTTGCCTCGT
CTTTGAAGGTTACTGCTCTATCTTTGTTCGCAGCATTCATTGCTGCTGCCATCTTGTGTTTCCTATACAAGACTTGTCTT
GCAGATTGCTATACGCAATACCGGACTACCGGTCTGAGCAGTACATCTTCATCTGGCTTTGGTCGAACCTCTGAGGCTTC
GACAGCTGTCCCGCGGACAGCGTCTGAGGTTTCTATTCCCTTAGGGGATAGATCTGCTACTCCATCTTCCTGTTTACCTA
CATCTGGTATTTCATCTATTTGATTTGTCATGACGTCTTCCTCGAAGAGGAAACGTGGATCTGGTAAGACGAAGACGCGC
AAGAAGGCGCGTTACACGAAGTGGACATCATCGAGAACAACGTCAGCTGCAGACTCTCTGCAGGTACAGACGTTCCTGTG
GGCTGAGGATCAGTCCTTCAATACTGGTGGAGGTTGCAGACTGCTAACCTCCTTCACGCGTGGTTCAGGAGAGAACCAAC
GCAAATCCCAGGAGACCATTACGTACAAGGTTGCCGTCAACCTTGGCATCTCCGCCTCCACTACCGTTCAGAAGTATTGC
CTGACCAGCCATCCTATATGTTGGCTGGTCTACGATAAGACGCCTGGGATTGCTGACCTGACTCCAACTGACATCTTTGA
TGTCCCGACTGGGTTGAACAACTGGCCTTCAACCTGGAAGGTCAAGCGCGAAGCATCTCACCGCTTCGTGGTGAAACGGC
GCTGGCCGTTTAGACTGTCCGTCAATGGCTCTACGTTCTCTGCAGATTATACGAAGCTGCCGGTGCCCAATACAGACAAC
CTGTGTACGATCAACAGGTTCGCCAAGGGACTTGGAGTGCGAACCGAATGGAAGGACACGGTTTCTGCTGACGCCTCCGA
CATCAAGGGCGGAGCCCTGTACATAGTATTAGCCCCGGCTAATGGGCTTGTATTCACAGCTAGAGGTGTCATTAAAGTGT
ACTTCAAGAGTGTGGGCAATCAGTAGCCCAATGTAATGAGCCCATAGGGCGATGAATAAAATGGCACATTTTATTATGTC
ATTATGACGAACGAGTACAACGAGAAAATTACATAATTGGTTTTTGTGGGTCGCGAGGGGGAACCCGGAGCACGCACCCA
AAAACACTAAACACACAAACCCAGAATATCTATAGGATACATGGCCGGCGCGTGCCGATAGTCACGCTGTAGCTGGCCCT
AAGCTTCGAAGAAGCTCTCCCCTGCATACATGTAATGTATGATACAGTTTGTCTCAAACCAGCCAACCTGACTAGGTTGC
ATTTGTTTGAGCCAGTCTTCGTCCTCGTTCACGAGGATTATTGATGGAACTCCATTCTTGAGCATACGTCGTTTACCATA
TTTTGGATTTACGGTAATGTCAAACTGGCATCCAACAAGCCCTTTCCAACAAGGAACGAACTTGAATGGTATGTCATCTA
TGACATTGTAAGTTGCGTTAGCAACTAAGTTGAGGAAGTCGACGCTGTGTTGCCAGTAGTTGTGAGCTCCCAGACTTCTA
GCCCAAGAAGTCTTTCCAGTTCTTGTTGGTCCGCAAATGTAGAGTGATCTTCTTCGTTCATCAGGACGTCTGTCCTGCTA
ATATCAGACATCCATTGGAGGTCAAGCCTCGCTTGGTCCTCCGTTACGCCATTATGTATGGAATATGCATAAGGACTTAC
AGAAAATAATTCTGCTTGGAGCCATGCTCCAATCACCGGATGTTCATCCTGTGCTGGCATCCCGAAGGGATCCATATATG
GCGTAGGAACCGAAGGGAATAAGCTCTCTGCGGAGTATTGGAATTGTTGCAACCTCGTTGCCCAATCGAATGGGAACGTA
TTCCTGACCATCGAGAGATAGTCCTCCTTGCAGGTACTGCTCTTGATTATCTCCGCCATTTTGGCGTCTTTAGTTGATGG
AGCATCAACCTTCTTCTTCTTCTGCTTGGTGCTTGCTTGGAAAACCCCCGTTTCTACGAAACATAAGGGGTTTTTTTTGA
TGTATGCTAACACCTTGTGTGGCATTCTTGCATTCTGCACGTTAGGATGAAATTCTTCGACGTCGAAGAATTTAGCACTC
TTGGTGCGAACATATTTCTTGCATTGTACCAGACAATGCAAATGGTAAGATCCATCTTGATGTTCTTCTCTTGCCACGTA
GACATACGTGGGTTCGAACCTTCGAAGAAGGCTTGAGAGGTGTTCTTGCATGAACACCGGATCAAGGTGGCACTTGCTGT
ACGTTAAGAAAATGTTCCTTGATCTTACCTCGAAGCATGCTGCGACCGCCCCTGGGGCGCCAACCTCCCCGCTCGGTGAG
CGGGGGGCGCCAGTTTGCCCCCTGGCGTCTGATGTCTCACTGACAAAGGAGGCCATCTCGTCAGAGGCTATCGTGGATAG
ACTCTCAGCCCTTGGATGATCTTTGAGATCCGAGGTTTTGAAAAACTCCCTTCCTAGACCCTTGCCCCTTGGCGCCCCCC
CCCCCGCGCCCTATCTTATAGGGCCTCGGGTGGGTGGCTTTTTTCTGGGCCGGGCCGGCGTAAAGATGGGCAAGCAA"""

def genome_strip(genome):
    return genome.replace("\n", "")


a,b,c = obtain_ori_candidates(genome_strip(genome))

print(a)
print(b)
print(c)

TypeError: _extract_candidates() missing 1 required positional argument: 'alignments'